# Sports Analytics: The Story Behind the Numbers

## A Narrative-Driven Analysis

This notebook tells the story hidden in our sports data - from the drama of home advantage to the quest for predicting the unpredictable.

---

## Chapter 1: The Home Advantage Mystery

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.visualization import SportsVisualizer, set_matplotlib_style

set_matplotlib_style()
viz = SportsVisualizer()

# Load data
matches = pd.read_csv('../data/raw/matches.csv', parse_dates=['date'])
teams = pd.read_csv('../data/raw/teams.csv')
players = pd.read_csv('../data/raw/players.csv')
performances = pd.read_csv('../data/raw/performances.csv', parse_dates=['date'])

### The Tale of Two Venues

Imagine two identical teams playing the same match. The only difference? One plays at home, the other away. 

Our data reveals a consistent pattern that has fascinated sports analysts for decades:

In [ ]:
# The home advantage story
results = matches['result'].value_counts(normalize=True)

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#3498db', '#95a5a6', '#e74c3c']
labels = ['Home Wins', 'Draws', 'Away Wins']
values = [results['H'], results['D'], results['A']]

wedges, texts, autotexts = ax.pie(values, labels=labels, autopct='%1.1f%%', 
                                   colors=colors, startangle=90,
                                   explode=[0.05, 0, 0])

ax.set_title('The Home Advantage: A Consistent Pattern', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

home_win_rate = results['H']
print(f"\n📖 The Story: Home teams win {home_win_rate:.1%} of all matches.")
print(f"   That's {(home_win_rate / 0.333 - 1) * 100:.0f}% more than random chance would predict.")

### But What Drives This Advantage?

Let's investigate the goal difference:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Goals distribution
ax1 = axes[0]
matches['home_score'].hist(bins=range(0, 8), ax=ax1, alpha=0.7, label='Home Goals', color='#3498db')
matches['away_score'].hist(bins=range(0, 8), ax=ax1, alpha=0.7, label='Away Goals', color='#e74c3c')
ax1.set_xlabel('Goals Scored')
ax1.set_ylabel('Frequency')
ax1.set_title('Goal Scoring: Home vs Away')
ax1.legend()

# Average by match outcome
ax2 = axes[1]
avg_goals = matches.groupby('result').agg({
    'home_score': 'mean',
    'away_score': 'mean'
}).rename(index={'H': 'Home Win', 'D': 'Draw', 'A': 'Away Win'})

x = np.arange(3)
width = 0.35

ax2.bar(x - width/2, avg_goals['home_score'], width, label='Home Goals', color='#3498db')
ax2.bar(x + width/2, avg_goals['away_score'], width, label='Away Goals', color='#e74c3c')
ax2.set_xticks(x)
ax2.set_xticklabels(avg_goals.index)
ax2.set_ylabel('Average Goals')
ax2.set_title('Average Goals by Match Outcome')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"\n📖 The Story: Home teams score an average of {matches['home_score'].mean():.2f} goals per match,")
print(f"   while away teams manage only {matches['away_score'].mean():.2f} goals.")
print(f"   That extra {matches['home_score'].mean() - matches['away_score'].mean():.2f} goals per game translates to significantly more wins.")

---

## Chapter 2: The Rise and Fall of Teams

Every season tells a story of triumph and struggle. Let's follow some teams through their journey.

In [ ]:
# Calculate team standings over time
def calculate_season_progress(matches, team_name):
    team_matches = matches[
        (matches['home_team'] == team_name) | (matches['away_team'] == team_name)
    ].sort_values('date')
    
    points = []
    cumulative = 0
    
    for _, match in team_matches.iterrows():
        if match['home_team'] == team_name:
            cumulative += match['home_points']
        else:
            cumulative += match['away_points']
        points.append(cumulative)
    
    return team_matches['date'].values, points

# Get top 5 teams by total points
home_pts = matches.groupby('home_team')['home_points'].sum()
away_pts = matches.groupby('away_team')['away_points'].sum()
total_pts = home_pts.add(away_pts, fill_value=0).sort_values(ascending=False)
top_teams = total_pts.head(5).index.tolist()

In [ ]:
# Plot season progress for top teams
fig, ax = plt.subplots(figsize=(14, 7))

colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12']

for team, color in zip(top_teams, colors):
    dates, points = calculate_season_progress(matches, team)
    ax.plot(range(len(points)), points, label=team, color=color, linewidth=2.5)

ax.set_xlabel('Match Number')
ax.set_ylabel('Cumulative Points')
ax.set_title('The Title Race: A Season-Long Battle', fontsize=14, fontweight='bold')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📖 The Story: The title race is a marathon, not a sprint.")
print(f"   {top_teams[0]} emerged victorious with {int(total_pts[top_teams[0]])} points,")
print(f"   but the gap to {top_teams[1]} was only {int(total_pts[top_teams[0]] - total_pts[top_teams[1]])} points.")

---

## Chapter 3: The Player Performance Puzzle

Not all heroes wear the same position. Let's explore how different positions contribute to success.

In [ ]:
# Player performance by position
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Goals by position
ax1 = axes[0]
pos_goals = performances.groupby('position')['goals'].mean().reindex(['GK', 'DEF', 'MID', 'FWD'])
bars = ax1.bar(pos_goals.index, pos_goals.values, color=['#1abc9c', '#3498db', '#9b59b6', '#e74c3c'])
ax1.set_xlabel('Position')
ax1.set_ylabel('Average Goals per Match')
ax1.set_title('Goal Scoring by Position')

for bar, val in zip(bars, pos_goals.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.2f}', ha='center', fontsize=11)

# Match ratings by position
ax2 = axes[1]
pos_ratings = performances.groupby('position')['match_rating'].mean().reindex(['GK', 'DEF', 'MID', 'FWD'])
bars2 = ax2.bar(pos_ratings.index, pos_ratings.values, color=['#1abc9c', '#3498db', '#9b59b6', '#e74c3c'])
ax2.set_xlabel('Position')
ax2.set_ylabel('Average Match Rating')
ax2.set_title('Match Rating by Position')
ax2.set_ylim(6, 7)

for bar, val in zip(bars2, pos_ratings.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.2f}', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

print("\n📖 The Story: Forwards score the goals, but every position has its role.")
print(f"   Forwards average {pos_goals['FWD']:.2f} goals per match,")
print(f"   while goalkeepers (perhaps surprisingly) have the most consistent ratings.")

### The Star Players

In [ ]:
# Top performers
player_stats = performances.groupby('player_id').agg({
    'match_rating': 'mean',
    'goals': 'sum',
    'assists': 'sum',
    'minutes_played': 'sum'
)

player_stats = player_stats.merge(players[['player_id', 'player_name', 'position']], on='player_id')
player_stats['goals_per_90'] = player_stats['goals'] / (player_stats['minutes_played'] / 90)

# Top 10 by rating
top_players = player_stats.nlargest(10, 'match_rating')

fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#e74c3c' if p == 'FWD' else '#3498db' if p == 'MID' else '#2ecc71' for p in top_players['position']]
bars = ax.barh(top_players['player_name'], top_players['match_rating'], color=colors)
ax.set_xlabel('Average Match Rating')
ax.set_title('Top 10 Players by Average Match Rating', fontsize=14, fontweight='bold')
ax.set_xlim(6.5, 7.5)

# Add position labels
for bar, pos in zip(bars, top_players['position']):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            pos, va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\n📖 The Story: Consistency is key to greatness.")
print(f"   {top_players.iloc[0]['player_name']} leads with an average rating of {top_players.iloc[0]['match_rating']:.2f}.")

---

## Chapter 4: The Unpredictable Nature of Sports

Perhaps the most fascinating aspect of sports is its inherent unpredictability.

In [ ]:
# Upset analysis
matches_with_ratings = matches.merge(
    teams[['team_id', 'overall_rating']].rename(columns={'team_id': 'home_team_id', 'overall_rating': 'home_rating'}),
    on='home_team_id'
).merge(
    teams[['team_id', 'overall_rating']].rename(columns={'team_id': 'away_team_id', 'overall_rating': 'away_rating'}),
    on='away_team_id'
)

# Define upsets: weaker team wins
matches_with_ratings['rating_diff'] = matches_with_ratings['home_rating'] - matches_with_ratings['away_rating']
matches_with_ratings['upset'] = (
    ((matches_with_ratings['rating_diff'] > 5) & (matches_with_ratings['result'] == 'A')) |
    ((matches_with_ratings['rating_diff'] < -5) & (matches_with_ratings['result'] == 'H'))
)

upset_rate = matches_with_ratings['upset'].mean()

fig, ax = plt.subplots(figsize=(10, 6))

# Win rate by rating difference
matches_with_ratings['rating_bin'] = pd.cut(
    matches_with_ratings['rating_diff'],
    bins=[-30, -10, -5, 0, 5, 10, 30],
    labels=['Much Weaker', 'Weaker', 'Slightly Weaker', 'Slightly Better', 'Better', 'Much Better']
)

win_rates = matches_with_ratings.groupby('rating_bin')['result'].apply(
    lambda x: (x == 'H').mean()
)

colors = ['#e74c3c' if x < 0.5 else '#3498db' for x in win_rates]
bars = ax.bar(range(len(win_rates)), win_rates, color=colors, edgecolor='black')
ax.set_xticks(range(len(win_rates)))
ax.set_xticklabels(win_rates.index, rotation=45, ha='right')
ax.set_ylabel('Home Win Rate')
ax.set_title('The Favorites Don\'t Always Win', fontsize=14, fontweight='bold')
ax.axhline(y=0.5, color='gray', linestyle='--', label='50% threshold')
ax.legend()

for bar, val in zip(bars, win_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.0%}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print(f"\n📖 The Story: Even when heavily favored, victory isn't guaranteed.")
print(f"   About {upset_rate:.0%} of matches end in upsets, where the weaker team triumphs.")
print("   This unpredictability is what makes sports beautiful.")

---

## Chapter 5: The Key to Prediction

What makes a match predictable? Let's identify the key factors.

In [ ]:
# Feature importance visualization (from our model)
from src.models import WinProbabilityModel
import joblib

try:
    # Load saved model
    model_data = joblib.load('../models/win_probability_model.joblib')
    
    # Get feature importance
    if hasattr(model_data['model'], 'feature_importances_'):
        importance = model_data['model'].feature_importances_  
        features = model_data['feature_names']
        
        importance_df = pd.DataFrame({
            'feature': features,
            'importance': importance
        }).sort_values('importance', ascending=True).tail(10)
        
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.barh(importance_df['feature'], importance_df['importance'], color='#3498db')
        ax.set_xlabel('Importance')
        ax.set_title('The Keys to Predicting Match Outcomes', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        print("\n📖 The Story: Team quality and home advantage dominate predictions.")
except:
    print("Run notebook 03_predictive_modeling.ipynb first to train the model.")

---

## Epilogue: The Beauty of Sports Data

Through this analysis, we've discovered:

1. **Home Advantage is Real** - A consistent 10-15% boost for the home team
2. **Team Quality Matters** - But doesn't guarantee victory
3. **Every Position Counts** - Different roles, different contributions
4. **Predictability Has Limits** - ~15% of matches are upsets

### The Final Word

> "In sports, as in life, the numbers tell a story - but they don't write the ending."

The true beauty of sports analytics lies not in predicting the future, but in understanding the patterns that make the beautiful game so fascinating.

In [ ]:
# Final visualization: The complete picture
fig = viz.plot_league_table_heatmap(matches.head(300))
fig.update_layout(title="The Complete Picture: Head-to-Head Results Matrix")
fig.show()